# influ-JSON — L1: entrenar LoRA de personaje (Colab gratis)

Consume el **pack L0** exportado del Studio (`*_lora_pack.zip`).

**Antes de empezar**
1. Runtime → Change runtime type → **GPU (T4)**.
2. Ten a mano el `.zip` del Studio (`🧬 Exportar pack de entrenamiento LoRA`).
3. Guía: `docs/lora/L1_COLAB.md` en el repo.

Cero costo de producto: no hay API de pago. Flux-dev requiere token HF + licencia aceptada.

## 0) Configuración

In [ ]:
# Opciones
USE_SCHNELL = False  # True = FLUX.1-schnell (sin licencia de pago); False = FLUX.1-dev
TRAIN_STEPS = 1500   # 1000–2000 típico en T4; baja si se acaba el tiempo de Colab
WORK = "/content/influ_lora"

import os
os.makedirs(WORK, exist_ok=True)
print("WORK =", WORK)
print("GPU:")
!nvidia-smi -L || echo "⚠ Sin GPU — Runtime → Change runtime type → T4"

## 1) Clonar ai-toolkit e instalar deps

Puede tardar varios minutos la primera vez.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
!pip install -q -r requirements.txt
print("ai-toolkit listo")

## 2) Subir el pack L0 (`*_lora_pack.zip`)

Ejecuta la celda y elige el ZIP exportado desde el Studio.

In [ ]:
from google.colab import files
import zipfile, shutil, glob

uploaded = files.upload()
assert uploaded, "Sube un *_lora_pack.zip"

zip_name = next(iter(uploaded))
zip_path = os.path.join(WORK, zip_name)
with open(zip_path, "wb") as f:
    f.write(uploaded[zip_name])

pack_dir = os.path.join(WORK, "pack")
if os.path.isdir(pack_dir):
    shutil.rmtree(pack_dir)
os.makedirs(pack_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(pack_dir)

print("Contenido del pack:")
for p in sorted(glob.glob(pack_dir + "/**/*", recursive=True))[:40]:
    if os.path.isfile(p):
        print(" -", os.path.relpath(p, pack_dir))

trigger_path = os.path.join(pack_dir, "trigger.txt")
trigger = open(trigger_path).readline().strip() if os.path.isfile(trigger_path) else "ohwx_person"
print("\nTrigger word:", trigger)

n_imgs = len(glob.glob(os.path.join(pack_dir, "dataset", "*.jpg"))) + len(
    glob.glob(os.path.join(pack_dir, "dataset", "*.png"))
)
print("Imágenes en dataset:", n_imgs)
if n_imgs < 8:
    print("⚠ Pocas imágenes. Ideal 15–30. Genera más variantes en el Studio y reexporta.")

## 3) Copiar dataset + config al ai-toolkit

Ajusta el YAML (pasos + modelo base).

In [ ]:
import yaml, re

dst_dataset = "/content/ai-toolkit/dataset"
dst_config_dir = "/content/ai-toolkit/config"
os.makedirs(dst_config_dir, exist_ok=True)

if os.path.isdir(dst_dataset):
    shutil.rmtree(dst_dataset)
shutil.copytree(os.path.join(pack_dir, "dataset"), dst_dataset)

src_yaml = os.path.join(pack_dir, "config", "ai-toolkit-flux.yaml")
assert os.path.isfile(src_yaml), "Falta config/ai-toolkit-flux.yaml en el pack"

raw = open(src_yaml).read()
# Quitar comentario YAML de cabecera si molesta al parser; el pack usa ---
docs = list(yaml.safe_load_all(raw))
cfg = [d for d in docs if d][-1]

proc = cfg["config"]["process"][0]
proc["train"]["steps"] = int(TRAIN_STEPS)
if USE_SCHNELL:
    proc["model"]["name_or_path"] = "black-forest-labs/FLUX.1-schnell"
else:
    proc["model"]["name_or_path"] = "black-forest-labs/FLUX.1-dev"

out_yaml = os.path.join(dst_config_dir, "influ_json_flux_lora.yaml")
with open(out_yaml, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Dataset →", dst_dataset)
print("Config  →", out_yaml)
print("Modelo  →", proc["model"]["name_or_path"])
print("Steps   →", TRAIN_STEPS)

## 4) Hugging Face (solo si usas FLUX.1-dev)

1. Crea token en https://huggingface.co/settings/tokens  
2. Acepta la licencia del modelo en la página del repo.  
3. Pega el token abajo (o usa el widget de login).

Si `USE_SCHNELL = True`, puedes saltarte esta celda.

In [ ]:
if not USE_SCHNELL:
    from huggingface_hub import login
    # Pega tu token o deja que el widget pida login:
    login()
else:
    print("Schnell: HF login opcional")

## 5) Entrenar

En T4 suele tardar ~30–90 min según steps. No cierres la pestaña.

In [ ]:
%cd /content/ai-toolkit
!python run.py config/influ_json_flux_lora.yaml

## 6) Descargar el `.safetensors`

Busca el último checkpoint en `output/` y descárgalo + el `trigger.txt`.

In [ ]:
from pathlib import Path

outs = sorted(Path("/content/ai-toolkit/output").rglob("*.safetensors"), key=lambda p: p.stat().st_mtime)
assert outs, "No se encontró ningún .safetensors — revisa el log del entrenamiento"
best = outs[-1]
print("LoRA:", best, "(", best.stat().st_size // (1024*1024), "MB)")

# Copia trigger junto al peso para L2
trigger_out = Path("/content") / f"{trigger}_TRIGGER.txt"
trigger_out.write_text(f"{trigger}\n\nUso: '{trigger} woman, <pose/escena>'\n")

files.download(str(best))
files.download(str(trigger_out))
print("Descargas iniciadas. Guarda el .safetensors fuera de Colab (Drive/local).")

## Siguiente

- **L2:** cargar el `.safetensors` en ComfyUI e invocar el trigger.  
- El Studio sigue generando con Pollinations + `character_lock` si no hay LoRA (free path intacto).  
- Plan: ROADMAP → Fase L.